In [31]:
from openadmet.toolkit.database.chembl import PermissiveChEMBLTargetCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm

In [32]:
def gather_chembl_data_for_target(target_name: str, chembl_tid: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = PermissiveChEMBLTargetCurator(chembl_target_id=chembl_tid, version=chembl_ver, standard_type="EC50", require_pchembl=True)
    activity_data = pctc.get_activity_data(return_as="df")

    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [33]:
targets = {
    "AHR": "CHEMBL3201",
    "PXR": "CHEMBL3401",
}

In [34]:
chembl_ver = 35

In [35]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

In [36]:
settings = S3Settings()

In [37]:
bucket = "openadmet-data-public-dev"

In [38]:
bucket = S3Bucket.from_settings(settings, bucket)

In [39]:
import datetime

In [40]:
t = datetime.datetime.now()

In [41]:
date = t.strftime("%Y-%m-%d")

In [42]:
location=f"ChEMBL{chembl_ver}_EC50"

In [43]:
import os
from pathlib import Path

location_path = Path(location)

In [44]:
location_path.mkdir(exist_ok=False)

In [45]:
uris_raw = {}
uris_agg = {}
for target, chembl_tid in targets.items():

    agg, raw  = gather_chembl_data_for_target(target, chembl_tid, chembl_ver)
    # TODO: make a function this is clunky
    fname_agg = f"ChEMBL_EC50_{target}_{chembl_tid}_aggregated.parquet"
    fname_raw = f"ChEMBL_EC50_{target}_{chembl_tid}_raw.parquet"
    
    agg.to_parquet(location_path/fname_agg)
    raw.to_parquet(location_path/fname_raw)
    
    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    # get S3 URIs
    uri_agg = bucket.to_uri(bucket_destination_agg)
    uris_agg[target] = uri_agg

    uri_raw = bucket.to_uri(bucket_destination_raw)
    uris_raw[target] = uri_raw

working on target AHR


100%|██████████| 276/276 [00:00<00:00, 6792.16it/s]


smiles duplicates 0
inchikey duplicates 0
working on target PXR


100%|██████████| 743/743 [00:00<00:00, 4516.82it/s]


smiles duplicates 0
inchikey duplicates 0


In [46]:
import intake
intake.Catalog?
cat = intake.entry.Catalog()

Init signature:
intake.Catalog(
    entries: 'Iterable[ReaderDescription] | Mapping | None' = None,
    aliases: 'dict[str, int] | None' = None,
    data: 'Iterable[DataDescription] | Mapping' = None,
    user_parameters: 'dict[str, BaseUserParameter] | None' = None,
    parameter_overrides: 'dict[str, Any] | None' = None,
    metadata: 'dict | None' = None,
)
Docstring:      A collection of data and reader descriptions.
File:           ~/miniconda3/envs/openadmet_toolkit/lib/python3.12/site-packages/intake/readers/entry.py
Type:           type
Subclasses:     THREDDSCatalog

In [47]:
uris_agg

{'AHR': 's3://openadmet-data-public-dev/ChEMBL35_EC50/ChEMBL_EC50_AHR_CHEMBL3201_aggregated.parquet',
 'PXR': 's3://openadmet-data-public-dev/ChEMBL35_EC50/ChEMBL_EC50_PXR_CHEMBL3401_aggregated.parquet'}

In [48]:
uris_raw

{'AHR': 's3://openadmet-data-public-dev/ChEMBL35_EC50/ChEMBL_EC50_AHR_CHEMBL3201_raw.parquet',
 'PXR': 's3://openadmet-data-public-dev/ChEMBL35_EC50/ChEMBL_EC50_PXR_CHEMBL3401_raw.parquet'}

In [49]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [50]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

In [51]:
catname = f"CATALOG_{location}.yaml"

In [52]:
cat.to_yaml_file(catname)

In [53]:
cat_location = location+ "/" +catname

In [54]:
cat_location

'ChEMBL35_EC50/CATALOG_ChEMBL35_EC50.yaml'

In [55]:
bucket.push_file(catname, cat_location)

In [56]:
cat_uri = bucket.to_uri(cat_location)